# 07_valid_error_analysis_oof

원본 Colab 셀에서 분리. (`#@title 74클래스 valid셋 오류 분석 (fold별 best → 자기 valid, OOF)`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [매 세션 3] dataset_74 zip 복원
import os                                                     # 경로 확인 도구

ZIP = os.path.join(PROJ_ROOT, "dataset_74_5fold.zip")         # 74클래스 5-fold zip(드라이브)
print("zip 존재:", os.path.exists(ZIP))                        # True 확인

!cp "$ZIP" /content/                                           # 드라이브 → 로컬 복사(속도 위해)
!unzip -qo /content/dataset_74_5fold.zip -d /content/dataset_74  # 압축 해제(-o=덮어쓰기)
print("복원 fold:", sorted(d for d in os.listdir("/content/dataset_74") if d.startswith("fold")))

In [ ]:
#@title 74클래스 valid셋 오류 분석 (fold별 best → 자기 valid, OOF)
#@markdown fold0~4 각각의 best로 자기 fold valid를 추론 → GT와 매칭해 미검출/분류오류/박스오류 3유형으로 분류하고, 틀린 항목 전부 출력 + 클래스별 집계 표 저장

# ============================================================
# [1] 설치
# ============================================================
!pip install -q "rfdetr[train,loggers]"                        # RF-DETR 추론 도구

# ============================================================
# [2] Drive 마운트 + import
# ============================================================
from google.colab import drive                                  # 코랩-드라이브 연결 도구
drive.mount("/content/drive", force_remount=True)               # 드라이브 마운트

import os, re, gc, json                                         # 환경변수·정규식·메모리·json 도구
import unicodedata                                              # NFC/NFD 차이 흡수 도구
import numpy as np                                               # 배열 계산 도구
import pandas as pd                                              # 결과 표 저장 도구
import torch                                                     # CUDA 메모리 정리 도구
from PIL import Image                                            # 이미지 열기 도구
from pathlib import Path                                         # 경로 객체 도구
from rfdetr import RFDETRMedium                                   # 모델 클래스

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # CUDA 메모리 단편화 완화

# ============================================================
# [3] 경로 + 설정
# ============================================================
TEAM_ROOT = Path("/content/drive/MyDrive/1팀 공유 문서")
USER_ROOT = TEAM_ROOT / "김태윤"
PROJ_ROOT = TEAM_ROOT / "ai12-level1-project"
LABEL_MAP_PATH = USER_ROOT / "label_map_74.json"
OUT_DIR = USER_ROOT / "error_analysis_74cls"                     # 분석 결과 저장 폴더
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_dir(parent, keyword):
    # NFC 정규화 비교로 폴더 탐색 (한글 폴더명 자모결합 차이 흡수)
    key = unicodedata.normalize("NFC", keyword)
    for name in os.listdir(parent):
        if key in unicodedata.normalize("NFC", name) and (parent / name).is_dir():
            return parent / name
    return None

outputs_dir = find_dir(USER_ROOT, "outputs_74")
assert outputs_dir is not None, f"outputs_74 폴더 못 찾음"
MODEL_DIR = find_dir(outputs_dir, "베스트모델")
assert MODEL_DIR is not None, f"베스트모델 폴더 못 찾음"
print("MODEL_DIR:", MODEL_DIR)

NUM_CLASSES = 74                                                 # 74클래스
RESOLUTION = 576                                                 # 학습과 동일
PRED_THRESHOLD = 0.30                                            # 오류분석용 threshold (제출과 달리 실사용 수준으로 컷)
IOU_MATCH_THR = 0.75                                             # 대회 metric 하한선(mAP@[0.75:0.95])과 동일

# ============================================================
# [4] dataset_74 복원 (세션 재시작 대비)
# ============================================================
DATASET = Path("/content/dataset_74")
if not DATASET.exists():                                          # 이미 있으면 건너뜀
    ZIP = PROJ_ROOT / "dataset_74_5fold.zip"
    assert ZIP.exists(), f"zip 없음: {ZIP}"
    !cp "{ZIP}" /content/
    !unzip -qo /content/dataset_74_5fold.zip -d /content/dataset_74
print("복원 fold:", sorted(d for d in os.listdir(DATASET) if d.startswith("fold")))

# ============================================================
# [5] 체크포인트 + 역매핑표
# ============================================================
ckpts = {}                                                        # {fold: 경로}
for p in MODEL_DIR.glob("checkpoint_best_total_*_74.pth"):
    m = re.search(r"total_(\d\d)_74", p.name)
    if m:
        ckpts[int(m.group(1))] = p
assert len(ckpts) == 5, f"best 5개 아님: {len(ckpts)}"
print("체크포인트:", {k: v.name for k, v in sorted(ckpts.items())})

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_map = json.load(f)
id2kcode = {int(k): int(v) for k, v in label_map.items()}         # 내부id -> K코드 (사람이 읽기 위함)

# ============================================================
# [6] IoU 계산 도구
# ============================================================
def iou_xyxy(a, b):
    # 박스 두 개(xyxy)의 IoU 계산 도구 #[x1,y1,x2,y2]
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])                   # 교집합 좌상단
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])                   # 교집합 우하단
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)             # 교집합 폭·높이 (음수면 0)
    inter = iw * ih                                                # 교집합 넓이
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter                                # 합집합 넓이
    return inter / union if union > 0 else 0.0

# ============================================================
# [7] fold별 best -> 자기 valid 추론 + GT 매칭
# ============================================================
errors = []                                                        # 틀린 항목 전부 누적
n_gt_total, n_correct = 0, 0                                       # 전체 GT 수 / 완전 정답 수

for fi, ckpt in sorted(ckpts.items()):
    valid_dir = DATASET / f"fold{fi}" / "valid"                    # 이 fold가 학습에 안 쓴 valid
    with open(valid_dir / "_annotations.coco.json", "r", encoding="utf-8") as f:
        coco = json.load(f)

    id2file = {im["id"]: im["file_name"] for im in coco["images"]} # image_id -> 파일명
    gt_by_image = {}                                                # 이미지별 GT 박스 모음
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        gt_by_image.setdefault(ann["image_id"], []).append({
            "bbox": [x, y, x + w, y + h],                           # xywh -> xyxy 변환
            "cat": ann["category_id"],
            "ann_id": ann["id"],
        })

    print(f"\n[fold {fi}] valid {len(coco['images'])}장 / GT {len(coco['annotations'])}개 — 모델 로드")
    model = RFDETRMedium(pretrain_weights=str(ckpt), num_classes=NUM_CLASSES, resolution=RESOLUTION)

    for img_id, fname in id2file.items():
        fpath = valid_dir / fname
        img = Image.open(fpath).convert("RGB")
        det = model.predict(img, threshold=PRED_THRESHOLD)          # 실사용 수준 threshold로 추론

        # 예측 목록 정리 (score 높은 순)
        preds = []
        if len(det) > 0:
            order = np.argsort(-det.confidence)
            for i in order:
                preds.append({
                    "bbox": det.xyxy[i].tolist(),
                    "cat": int(det.class_id[i]),
                    "score": float(det.confidence[i]),
                    "used": False,                                   # 이미 GT에 매칭됐는지 표시
                })

        gts = gt_by_image.get(img_id, [])
        n_gt_total += len(gts)

        for gt in gts:                                              # GT 하나씩 최적 예측 찾기
            best_iou, best_pred = 0.0, None
            for p in preds:
                if p["used"]:
                    continue                                        # 이미 다른 GT에 쓰인 예측은 제외
                iou = iou_xyxy(gt["bbox"], p["bbox"])
                if iou > best_iou:
                    best_iou, best_pred = iou, p

            # --- 3유형 분류 ---
            if best_pred is None or best_iou < 0.1:                 # 겹치는 예측 자체가 없음
                errors.append({
                    "fold": fi, "file_name": fname, "ann_id": gt["ann_id"],
                    "error_type": "미검출(FN)",
                    "gt_cat": gt["cat"], "gt_kcode": id2kcode[gt["cat"]],
                    "pred_cat": None, "pred_kcode": None,
                    "iou": round(best_iou, 3), "score": None,
                })
            elif best_iou >= IOU_MATCH_THR and best_pred["cat"] == gt["cat"]:
                best_pred["used"] = True                            # 완전 정답
                n_correct += 1
            elif best_iou >= IOU_MATCH_THR:                         # 박스는 맞고 클래스 틀림
                best_pred["used"] = True
                errors.append({
                    "fold": fi, "file_name": fname, "ann_id": gt["ann_id"],
                    "error_type": "분류오류",
                    "gt_cat": gt["cat"], "gt_kcode": id2kcode[gt["cat"]],
                    "pred_cat": best_pred["cat"], "pred_kcode": id2kcode[best_pred["cat"]],
                    "iou": round(best_iou, 3), "score": round(best_pred["score"], 3),
                })
            else:                                                   # 겹치긴 하나 IoU 0.75 미달
                best_pred["used"] = True
                same = "동일클래스" if best_pred["cat"] == gt["cat"] else "클래스도틀림"
                errors.append({
                    "fold": fi, "file_name": fname, "ann_id": gt["ann_id"],
                    "error_type": f"박스오류({same})",
                    "gt_cat": gt["cat"], "gt_kcode": id2kcode[gt["cat"]],
                    "pred_cat": best_pred["cat"], "pred_kcode": id2kcode[best_pred["cat"]],
                    "iou": round(best_iou, 3), "score": round(best_pred["score"], 3),
                })

        # 남은 예측 = 오탐(FP)
        for p in preds:
            if not p["used"]:
                errors.append({
                    "fold": fi, "file_name": fname, "ann_id": None,
                    "error_type": "오탐(FP)",
                    "gt_cat": None, "gt_kcode": None,
                    "pred_cat": p["cat"], "pred_kcode": id2kcode[p["cat"]],
                    "iou": None, "score": round(p["score"], 3),
                })

    del model; gc.collect(); torch.cuda.empty_cache()               # 다음 fold 대비 GPU 정리
    print(f"[fold {fi}] 완료")

# ============================================================
# [8] 결과 집계 + 출력
# ============================================================
err_df = pd.DataFrame(errors)

print(f"\n{'='*60}")
print(f"전체 GT: {n_gt_total}개 / 완전 정답: {n_correct}개 ({n_correct/n_gt_total*100:.1f}%)")
print(f"오류 항목 총: {len(err_df)}개")
print(f"{'='*60}\n")

print("=== 오류 유형별 개수 ===")
print(err_df["error_type"].value_counts().to_string())

print("\n=== 클래스별 오류 수 (상위 15) ===")
cls_err = err_df[err_df["gt_kcode"].notna()].groupby("gt_kcode").size().sort_values(ascending=False)
print(cls_err.head(15).to_string())

print("\n=== 분류오류 혼동 쌍 (GT K코드 -> 예측 K코드) ===")
conf = err_df[err_df["error_type"] == "분류오류"].groupby(["gt_kcode", "pred_kcode"]).size().sort_values(ascending=False)
print(conf.to_string() if len(conf) else "분류오류 없음")

print("\n=== 틀린 항목 전체 ===")
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
print(err_df.to_string(index=False))

out_csv = OUT_DIR / "valid_errors_74cls.csv"
err_df.to_csv(out_csv, index=False, encoding="utf-8-sig")           # 엑셀에서 한글 안 깨지게 저장
print(f"\n저장 완료: {out_csv}")